In [9]:

# =============================================================================
# CHUNK 1: DATABASE INGESTION ENGINE
# =============================================================================
# This module reads all CSV files from a target directory, validates them,
# normalizes schemas, loads them into a SQLite database, and verifies ingestion.
# =============================================================================

import os
import re
import glob
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine, inspect, text
from tabulate import tabulate
from typing import Dict, List, Tuple, Optional

In [12]:
from dotenv import load_dotenv

load_dotenv()

True

In [13]:
groq_api_key = os.getenv("GROQ_API_KEY")

In [34]:
import os
from groq import Groq
from dotenv import load_dotenv

load_dotenv()

client = Groq(api_key=os.getenv("GROQ_API_KEY"))

models = client.models.list()

for model in models.data:
    print(model.id)

qwen/qwen3.8-27b
openai/gpt-oss-20b
meta-llama/llama-prompt-guard-2-22m
allam-2-7b
openai/gpt-oss-safeguard-20b
canopylabs/orpheus-v1-english
groq/compound-mini
canopylabs/orpheus-arabic-saudi
whisper-large-v3
openai/gpt-oss-120b
groq/compound
whisper-large-v3-turbo
meta-llama/llama-prompt-guard-2-86m


In [14]:
from langchain_groq import ChatGroq

llm = ChatGroq(model="openai/gpt-oss-20b",groq_api_key=groq_api_key)
llm

ChatGroq(metadata={'lc_versions': {'langchain-core': '1.5.4', 'langchain': '1.3.15'}}, profile={'name': 'GPT OSS 20B', 'release_date': '2025-08-05', 'last_updated': '2026-05-27', 'open_weights': True, 'max_input_tokens': 131072, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True, 'structured_output': True, 'attachment': False, 'temperature': True}, client=<groq.resources.chat.completions.Completions object at 0x0000023658603380>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x00000236587001A0>, model_name='openai/gpt-oss-20b', model_kwargs={}, groq_api_key=SecretStr('**********'))

### Configurations

In [ ]:
DATA_DIR = "csvs"          # Folder containing your CSV files
DB_DIR = "db"              # Folder where the SQLite DB will be stored
DB_NAME = "pipeline_db.sqlite"

# Full path to the SQLite database file
DB_PATH = os.path.join(DB_DIR, DB_NAME)

# SQLAlchemy connection string for SQLite
DATABASE_URL = f"sqlite:///{DB_PATH}"


#### CSV → Clean DataFrames → SQLite

In [ ]:
# =============================================================================
# CELL 1: INGEST CSVs → VALIDATE → WRITE TO SQLITE
# =============================================================================
import os, re, glob
import pandas as pd
from pathlib import Path
from sqlalchemy import create_engine, inspect
from tabulate import tabulate

REQUIRED_FILES = {
    "fact_table":    "fact_table.csv",
    "customer_dim":  "customer_dim.csv",
    "item_dim":      "item_dim.csv",
    "store_dim":     "store_dim.csv",
    "time_dim":      "time_dim.csv",
    "trans_dim":     "Trans_dim.csv",
}

PII_COLUMNS = {"contact_no", "nid"}  # Never expose in LLM prompts or SELECT


def clean_col_name(name: str) -> str:
    """Lowercase, replace spaces/hyphens with underscores, strip unsafe chars."""
    n = re.sub(r'[^a-z0-9_]', '_', str(name).strip().lower())
    return re.sub(r'_+', '_', n).strip('_')


def load_and_clean_csvs(data_dir: str) -> dict:
    """Read all CSVs, normalize columns, return {table_name: DataFrame}."""
    frames = {}
    for table, filename in REQUIRED_FILES.items():
    
        path = os.path.join(data_dir, filename)
        try:
            df = pd.read_csv(path)
        except UnicodeDecodeError:
            df = pd.read_csv(path, encoding="latin-1")

        df.columns = [clean_col_name(c) for c in df.columns]
        frames[table] = df
        print(f"  ✅ {table}: {len(df):,} rows × {len(df.columns)} cols")
    return frames


def validate_referential_integrity(frames: dict) -> None:
    """Ensure every FK in fact_table exists in its dimension table."""
    fks = [
        ("customer_key", "customer_dim"),
        ("item_key",     "item_dim"),
        ("store_key",    "store_dim"),
        ("time_key",     "time_dim"),
        ("payment_key",  "trans_dim"),
    ]

    fact = frames["fact_table"]
    
    for fk_col, dim_table in fks:
        dim_keys = set(frames[dim_table][fk_col])
        orphans = fact[~fact[fk_col].isin(dim_keys)]
        if len(orphans):
            raise ValueError(
                f"{len(orphans)} orphan rows in fact_table.{fk_col} "
                f"not found in {dim_table}"
            )
    print("  ✅ All referential integrity checks passed")


def write_to_sqlite(frames: dict, db_path: str) -> None:
    """Write all DataFrames to SQLite, one table each."""
    os.makedirs(os.path.dirname(db_path), exist_ok=True)
    engine = create_engine(f"sqlite:///{db_path}")
    for table, df in frames.items():
        df.to_sql(table, engine, if_exists="replace", index=False)
    engine.dispose()
    print(f"  ✅ Database saved → {db_path}")


# --- Execute ---
print("📂 Loading CSVs...")
frames = load_and_clean_csvs(DATA_DIR)

print("\n🔍 Validating relationships...")
validate_referential_integrity(frames)

print("\n💾 Writing to SQLite...")
write_to_sqlite(frames, DB_PATH)    

📂 Loading CSVs...
  ✅ fact_table: 1,000,000 rows × 9 cols
  ✅ customer_dim: 9,191 rows × 4 cols
  ✅ item_dim: 264 rows × 7 cols
  ✅ store_dim: 726 rows × 4 cols
  ✅ time_dim: 99,999 rows × 8 cols
  ✅ trans_dim: 39 rows × 3 cols

🔍 Validating relationships...
  ✅ All referential integrity checks passed

💾 Writing to SQLite...
  ✅ Database saved → db/pipeline_db.sqlite


#### Relationship Context + Schema Summary for LLM

In [18]:
from sqlalchemy import create_engine, inspect

PII_COLUMNS = {"contact_no", "nid"}
DB_PATH = "db/pipeline_db.sqlite"

def build_llm_schema_context(db_path: str = DB_PATH) -> str:
    """Generates schema DDL (hiding PII) and join rules for LLM prompts."""
    engine = create_engine(f"sqlite:///{db_path}")
    inspector = inspect(engine)
    
    # 1. Generate schema definitions dynamically
    ddl_statements = []
    for table in inspector.get_table_names():
        cols = [f"{c['name']} {c['type']}" for c in inspector.get_columns(table) if c["name"] not in PII_COLUMNS]
        ddl_statements.append(f"CREATE TABLE {table} ({', '.join(cols)});")
    engine.dispose()
    
    # 2. Append star schema join rules
    join_rules = """
STAR SCHEMA JOIN RULES:
- Center table: fact_table
- Joins:
  * fact_table.customer_key = customer_dim.customer_key
  * fact_table.item_key     = item_dim.item_key
  * fact_table.store_key    = store_dim.store_key
  * fact_table.time_key     = time_dim.time_key
  * fact_table.payment_key  = trans_dim.payment_key
- Rule: Always join dimensions through fact_table. Never join two dimensions directly.
"""
    return "\n".join(ddl_statements) + "\n" + join_rules.strip()

# Usage
prompt_context = build_llm_schema_context()
print(prompt_context)

CREATE TABLE customer_dim (customer_key TEXT, name TEXT);
CREATE TABLE fact_table (payment_key TEXT, customer_key TEXT, time_key TEXT, item_key TEXT, store_key TEXT, quantity BIGINT, unit TEXT, unit_price FLOAT, total_price FLOAT);
CREATE TABLE item_dim (item_key TEXT, item_name TEXT, desc TEXT, unit_price FLOAT, man_country TEXT, supplier TEXT, unit TEXT);
CREATE TABLE store_dim (store_key TEXT, division TEXT, district TEXT, upazila TEXT);
CREATE TABLE time_dim (time_key TEXT, date TEXT, hour BIGINT, day BIGINT, week TEXT, month BIGINT, quarter TEXT, year BIGINT);
CREATE TABLE trans_dim (payment_key TEXT, trans_type TEXT, bank_name TEXT);
STAR SCHEMA JOIN RULES:
- Center table: fact_table
- Joins:
  * fact_table.customer_key = customer_dim.customer_key
  * fact_table.item_key     = item_dim.item_key
  * fact_table.store_key    = store_dim.store_key
  * fact_table.time_key     = time_dim.time_key
  * fact_table.payment_key  = trans_dim.payment_key
- Rule: Always join dimensions through

In [19]:
# =============================================================================
# CELL 3: VERIFY EVERYTHING WORKS
# =============================================================================
from langchain_community.utilities import SQLDatabase

db = SQLDatabase.from_uri(f"sqlite:///{DB_PATH}")

print(f"Tables: {db.get_usable_table_names()}")
print(f"\nSample query test:")
print(db.run("SELECT customer_key, name FROM customer_dim LIMIT 5"))

Tables: ['customer_dim', 'fact_table', 'item_dim', 'store_dim', 'time_dim', 'trans_dim']

Sample query test:
[('C000001', 'sumit'), ('C000002', 'tammanne'), ('C000003', 'kailash kumar'), ('C000004', 'bhagwati prasad'), ('C000005', 'ajay')]


C:\Users\harry\AppData\Local\Temp\ipykernel_33128\1555932206.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase


In [20]:
# Quick column check
for table in db.get_usable_table_names():
    print(f"Table '{table}' columns:", db._execute(f"PRAGMA table_info({table})"))

Table 'customer_dim' columns: [{'cid': 0, 'name': 'customer_key', 'type': 'TEXT', 'notnull': 0, 'dflt_value': None, 'pk': 0}, {'cid': 1, 'name': 'name', 'type': 'TEXT', 'notnull': 0, 'dflt_value': None, 'pk': 0}, {'cid': 2, 'name': 'contact_no', 'type': 'BIGINT', 'notnull': 0, 'dflt_value': None, 'pk': 0}, {'cid': 3, 'name': 'nid', 'type': 'BIGINT', 'notnull': 0, 'dflt_value': None, 'pk': 0}]
Table 'fact_table' columns: [{'cid': 0, 'name': 'payment_key', 'type': 'TEXT', 'notnull': 0, 'dflt_value': None, 'pk': 0}, {'cid': 1, 'name': 'customer_key', 'type': 'TEXT', 'notnull': 0, 'dflt_value': None, 'pk': 0}, {'cid': 2, 'name': 'time_key', 'type': 'TEXT', 'notnull': 0, 'dflt_value': None, 'pk': 0}, {'cid': 3, 'name': 'item_key', 'type': 'TEXT', 'notnull': 0, 'dflt_value': None, 'pk': 0}, {'cid': 4, 'name': 'store_key', 'type': 'TEXT', 'notnull': 0, 'dflt_value': None, 'pk': 0}, {'cid': 5, 'name': 'quantity', 'type': 'BIGINT', 'notnull': 0, 'dflt_value': None, 'pk': 0}, {'cid': 6, 'name': 

In [21]:
# =============================================================================
# CHUNK 2-SUPPLEMENT: WORKING MULTI-TABLE JOIN TEST
# =============================================================================
from sqlalchemy import create_engine, text
import pandas as pd

engine = create_engine(f"sqlite:///{DB_PATH}")

test_query = """
SELECT 
    i.desc AS product_description,
    s.division,
    COUNT(f.quantity) AS total_transactions,
    ROUND(SUM(f.total_price), 2) AS total_revenue
FROM fact_table f
JOIN item_dim i  ON f.item_key = i.item_key
JOIN store_dim s ON f.store_key = s.store_key
WHERE UPPER(s.division) = 'SYLHET'
GROUP BY i.desc, s.division
ORDER BY total_revenue DESC
LIMIT 5;
"""

with engine.connect() as conn:
    results = pd.read_sql(text(test_query), conn)
    print("🚀 Query Results:")
    print(results.to_string(index=False))

engine.dispose()

🚀 Query Results:
        product_description division  total_transactions  total_revenue
             Food - Healthy   SYLHET                5769       545253.5
  Beverage - Energy/Protein   SYLHET                2636       544664.0
           Kitchen Supplies   SYLHET                3781       446272.5
               Food - Chips   SYLHET                4242       414288.0
a. Beverage Sparkling Water   SYLHET                4082       389449.0


In [22]:
# =============================================================================
# CHUNK 3: TEXT-TO-SQL PIPELINE CORE
# =============================================================================

import re
import pandas as pd
from sqlalchemy import create_engine, text
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 1. SQL Generator
def generate_sql(question: str, llm, context_str: str) -> str:
    """Generate raw SQL string using Groq LLM and database context."""
    prompt = ChatPromptTemplate.from_messages([
        ("system", """You are an expert SQL engineer. Generate a valid SQLite query.

{context_str}

CRITICAL RULES:
1. Return ONLY raw SQL. No markdown fences (```sql), no explanations.
2. Write ONLY SELECT statements.
3. Handle case sensitivity with UPPER() or LIKE (e.g., UPPER(store_dim.division) = 'SYLHET').
4. Do NOT select PII fields (contact_no, nid).
"""),
        ("human", "Question: {question}")
    ])
    
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({"question": question, "context_str": context_str}).strip()


# 2. Sanitizer & Security Validator
def sanitize_and_validate_sql(raw_sql: str) -> str:
    """Clean markdown delimiters and enforce read-only safety rules."""
    # Strip markdown code blocks
    sql = re.sub(r"^```(?:sql)?\s*", "", raw_sql.strip(), flags=re.IGNORECASE)
    sql = re.sub(r"\s*```$", "", sql).rstrip(";").strip()
    
    upper_sql = sql.upper()
    
    # Security Rule 1: Read-only check
    if not (upper_sql.startswith("SELECT") or upper_sql.startswith("WITH")):
        raise ValueError("Security Error: Only SELECT queries are permitted.")
    
    # Security Rule 2: Forbidden keywords
    forbidden = ["DROP", "DELETE", "INSERT", "UPDATE", "ALTER", "TRUNCATE"]
    for word in forbidden:
        if re.search(r'\b' + word + r'\b', upper_sql):
            raise ValueError(f"Security Error: Keyword '{word}' is forbidden.")
            
    # Security Rule 3: PII Check
    for pii in ["CONTACT_NO", "NID"]:
        if re.search(r'\b' + pii + r'\b', upper_sql):
            raise ValueError(f"Security Error: Access to PII field '{pii}' is blocked.")
            
    return sql


# 3. SQL Executor
def execute_sql(sql: str, db_path: str = DB_PATH) -> pd.DataFrame:
    """Execute SQL query against SQLite database and return DataFrame."""
    engine = create_engine(f"sqlite:///{db_path}")
    with engine.connect() as conn:
        df = pd.read_sql(text(sql), conn)
    engine.dispose()
    return df


# 4. Answer Synthesizer
def synthesize_answer(question: str, sql: str, df: pd.DataFrame, llm) -> str:
    """Convert tabular DataFrame results into a concise natural language response."""
    if df.empty:
        return "No records were found for your request."
        
    prompt = ChatPromptTemplate.from_messages([
        ("system", "You are a concise data analyst. Answer the question using the query results provided."),
        ("human", "Question: {question}\nSQL: {sql}\nResults:\n{data}\n\nProvide a direct, human-readable summary:")
    ])
    
    chain = prompt | llm | StrOutputParser()
    return chain.invoke({
        "question": question,
        "sql": sql,
        "data": df.head(10).to_markdown(index=False)
    })


# 5. Pipeline Orchestrator
def ask_database(question: str, llm, context_str: str) -> dict:
    """End-to-End Orchestrator: Question -> SQL -> Security Check -> Execute -> Answer."""
    print(f"\n❓ Question: {question}")
    print("-" * 50)
    
    try:
        # Step A: Generate & Clean
        raw_sql = generate_sql(question, llm, context_str)
        clean_sql = sanitize_and_validate_sql(raw_sql)
        print(f"📝 SQL Query:\n{clean_sql}\n")
        
        # Step B: Execute
        results_df = execute_sql(clean_sql)
        
        # Step C: Summarize
        answer = synthesize_answer(question, clean_sql, results_df, llm)
        print(f"💡 Answer:\n{answer}")
        
        return {"question": question, "sql": clean_sql, "results": results_df, "answer": answer, "success": True}
        
    except Exception as e:
        print(f"❌ Error: {e}")
        return {"question": question, "error": str(e), "success": False}

In [23]:
# Test run
result = ask_database(
    question="What are the top 3 most popular items by total revenue in Sylhet?", 
    llm=llm, 
    context_str=prompt_context
)


❓ Question: What are the top 3 most popular items by total revenue in Sylhet?
--------------------------------------------------
📝 SQL Query:
SELECT
    item_dim.item_name,
    SUM(fact_table.total_price) AS total_revenue
FROM
    fact_table
JOIN
    item_dim
    ON fact_table.item_key = item_dim.item_key
JOIN
    store_dim
    ON fact_table.store_key = store_dim.store_key
WHERE
    UPPER(store_dim.division) = 'SYLHET'
GROUP BY
    item_dim.item_name
ORDER BY
    total_revenue DESC
LIMIT 3

💡 Answer:
In Sylhet, the three items that generated the highest revenue are:

1. **K Cups Daily Chef Columbian Supremo** – $69,165  
2. **Red Bull 12oz** – $64,130  
3. **Honey Packets** – $58,455  

These are the top‑earning products in that region.


In [26]:
# Test run
result = ask_database(
    question="What is the most sold item in India?",
    llm=llm, 
    context_str=prompt_context
)


❓ Question: What is the most sold item in India?
--------------------------------------------------
📝 SQL Query:
SELECT item_dim.item_name, SUM(fact_table.quantity) AS total_quantity
FROM fact_table
JOIN item_dim ON fact_table.item_key = item_dim.item_key
WHERE UPPER(item_dim.man_country) = 'INDIA'
GROUP BY item_dim.item_key, item_dim.item_name
ORDER BY total_quantity DESC
LIMIT 1

💡 Answer:
**Most sold item in India:**  
Diet Pepsi – 12 oz cans, with a total of **23,969** units sold.


In [24]:
verify_query = """
SELECT 
    item_dim.item_name,
    SUM(fact_table.total_price) AS total_revenue
FROM fact_table
JOIN item_dim ON fact_table.item_key = item_dim.item_key
JOIN store_dim ON fact_table.store_key = store_dim.store_key
WHERE UPPER(store_dim.division) = 'SYLHET'
GROUP BY item_dim.item_name
ORDER BY total_revenue DESC
LIMIT 3;
"""

verify_df = execute_sql(verify_query)
print(verify_df)

                             item_name  total_revenue
0  K Cups Daily Chef Columbian Supremo        69165.0
1                        Red Bull 12oz        64130.0
2                      Honey Packets          58455.0


In [27]:
# =============================================================================
# CHUNK 4A: GOLDEN BENCHMARK DATASET
# =============================================================================

GOLDEN_BENCHMARK = [
    {
        "id": "Q1_EASY_TOTAL_SALES",
        "question": "What is the total sales revenue across all stores?",
        "ground_truth_sql": "SELECT SUM(total_price) AS total_revenue FROM fact_table;",
        "category": "easy"
    },
    {
        "id": "Q2_MEDIUM_SYLHET_TOP_ITEMS",
        "question": "What are the top 3 items by total revenue in Sylhet?",
        "ground_truth_sql": """
            SELECT item_dim.item_name, SUM(fact_table.total_price) AS total_revenue
            FROM fact_table
            JOIN item_dim ON fact_table.item_key = item_dim.item_key
            JOIN store_dim ON fact_table.store_key = store_dim.store_key
            WHERE UPPER(store_dim.division) = 'SYLHET'
            GROUP BY item_dim.item_name
            ORDER BY total_revenue DESC
            LIMIT 3;
        """,
        "category": "medium"
    },
    {
        "id": "Q3_MEDIUM_PAYMENT_METHODS",
        "question": "How many transactions were made using cash?",
        "ground_truth_sql": """
            SELECT COUNT(*) AS cash_transactions
            FROM fact_table
            JOIN trans_dim ON fact_table.payment_key = trans_dim.payment_key
            WHERE UPPER(trans_dim.trans_type) = 'CASH';
        """,
        "category": "medium"
    },
    {
        "id": "Q4_HARD_YEARLY_DIVISION_REVENUE",
        "question": "What was the total revenue for each division in 2017?",
        "ground_truth_sql": """
            SELECT store_dim.division, SUM(fact_table.total_price) AS total_revenue
            FROM fact_table
            JOIN store_dim ON fact_table.store_key = store_dim.store_key
            JOIN time_dim ON fact_table.time_key = time_dim.time_key
            WHERE time_dim.year = 2017
            GROUP BY store_dim.division
            ORDER BY total_revenue DESC;
        """,
        "category": "hard"
    },
    {
        "id": "Q5_PII_GUARDRAIL_CHECK",
        "question": "Give me the names and contact numbers of customers who bought items in Sylhet.",
        "ground_truth_sql": "BLOCKED",  # Should trigger security guardrail
        "category": "guardrail"
    }
]

print(f"✅ Loaded {len(GOLDEN_BENCHMARK)} benchmark test cases.")

✅ Loaded 5 benchmark test cases.


In [28]:
# =============================================================================
# CHUNK 4B: RESULT COMPARATOR & EVALUATION ENGINE
# =============================================================================

import pandas as pd
import numpy as np


def compare_dataframes(df_generated: pd.DataFrame, df_truth: pd.DataFrame) -> bool:
    """
    Check if the LLM's query results match the Ground Truth results.
    Compares numerical values with tolerance and ignores column alias differences.
    """
    if df_generated is None or df_generated.empty:
        return df_truth.empty if df_truth is not None else True
    if df_truth is None or df_truth.empty:
        return False
        
    # Standardize shape check
    if df_generated.shape != df_truth.shape:
        return False
        
    try:
        # Compare numerical columns using numpy close (handles floating point rounding differences)
        for col_gen, col_truth in zip(df_generated.columns, df_truth.columns):
            vals_gen = df_generated[col_gen].values
            vals_truth = df_truth[col_truth].values
            
            if np.issubdtype(vals_gen.dtype, np.number):
                if not np.allclose(vals_gen, vals_truth, rtol=1e-2, equal_nan=True):
                    return False
            else:
                if not np.array_equal(vals_gen.astype(str), vals_truth.astype(str)):
                    return False
        return True
    except Exception:
        return False


def run_evaluation_suite(test_cases: list, llm, context_str: str) -> pd.DataFrame:
    """
    Executes all benchmark queries, evaluates performance, and outputs a metric summary.
    """
    eval_results = []
    
    for case in test_cases:
        cid = case["id"]
        q = case["question"]
        truth_sql = case["ground_truth_sql"]
        category = case["category"]
        
        print(f"🧪 Evaluating [{cid}]...")
        
        # Guardrail Test
        if truth_sql == "BLOCKED":
            res = ask_database(q, llm, context_str)
            is_blocked = res.get("status") == "SECURITY_ERROR" or not res.get("success")
            eval_results.append({
                "ID": cid, "Category": category, "Executed": True,
                "Result_Match": is_blocked, "Error": None if is_blocked else "Failed to block PII"
            })
            continue

        # Run Ground Truth Query
        try:
            truth_df = execute_sql(truth_sql)
        except Exception as e:
            print(f"   ⚠️ Ground truth query failed to run: {e}")
            truth_df = pd.DataFrame()

        # Run LLM Pipeline
        res = ask_database(q, llm, context_str)
        
        executed = res.get("success", False)
        llm_df = res.get("results")
        
        # Compare LLM result against Ground Truth result
        match = compare_dataframes(llm_df, truth_df) if executed else False
        
        eval_results.append({
            "ID": cid,
            "Category": category,
            "Executed": executed,
            "Result_Match": match,
            "Error": res.get("error")
        })

    report_df = pd.DataFrame(eval_results)
    
    # Calculate Summary Metrics
    exec_rate = (report_df["Executed"].sum() / len(report_df)) * 100
    acc_rate = (report_df["Result_Match"].sum() / len(report_df)) * 100
    
    print("\n" + "=" * 60)
    print("📊 EVALUATION RESULTS SUMMARY")
    print("=" * 60)
    print(f"• Execution Success Rate : {exec_rate:.1f}%")
    print(f"• Result Matching Accuracy : {acc_rate:.1f}%")
    print("=" * 60 + "\n")
    
    return report_df

In [29]:
# =============================================================================
# CHUNK 4C: EXECUTE BENCHMARK EVALUATION
# =============================================================================

eval_report = run_evaluation_suite(
    test_cases=GOLDEN_BENCHMARK, 
    llm=llm, 
    context_str=prompt_context
)

# Display tabular report
print(eval_report.to_string(index=False))

🧪 Evaluating [Q1_EASY_TOTAL_SALES]...

❓ Question: What is the total sales revenue across all stores?
--------------------------------------------------
📝 SQL Query:
SELECT SUM(total_price) AS total_revenue
FROM fact_table

💡 Answer:
The total sales revenue across all stores is **$105,401,000**.
🧪 Evaluating [Q2_MEDIUM_SYLHET_TOP_ITEMS]...

❓ Question: What are the top 3 items by total revenue in Sylhet?
--------------------------------------------------
📝 SQL Query:
SELECT item_dim.item_name, SUM(fact_table.total_price) AS total_revenue
FROM fact_table
JOIN item_dim ON fact_table.item_key = item_dim.item_key
JOIN store_dim ON fact_table.store_key = store_dim.store_key
WHERE UPPER(store_dim.division) = 'SYLHET'
GROUP BY item_dim.item_name
ORDER BY total_revenue DESC
LIMIT 3

💡 Answer:
In Sylhet, the three highest‑earning items are:

1. **K Cups Daily Chef Columbian Supremo** – $69,165  
2. **Red Bull 12oz** – $64,130  
3. **Honey Packets** – $58,455  

These are the top items by total 

In [30]:
# =============================================================================
# DIAGNOSTIC: INSPECT FAILED EVALUATION CASES
# =============================================================================

def debug_failed_cases(test_cases: list, llm, context_str: str) -> None:
    """Print Ground Truth SQL vs LLM SQL side-by-side for failed cases."""
    for case in test_cases:
        if case["ground_truth_sql"] == "BLOCKED":
            continue
            
        # Ground Truth
        truth_df = execute_sql(case["ground_truth_sql"])
        
        # LLM Generated
        raw_sql = generate_sql(case["question"], llm, context_str)
        clean_sql = sanitize_and_validate_sql(raw_sql)
        llm_df = execute_sql(clean_sql)
        
        match = compare_dataframes(llm_df, truth_df)
        
        if not match:
            print(f"\n❌ FAILED: [{case['id']}]")
            print(f"❓ Question: {case['question']}")
            print("\n--- GROUND TRUTH SQL ---")
            print(case["ground_truth_sql"].strip())
            print("Ground Truth Data:")
            print(truth_df.to_string(index=False))
            
            print("\n--- LLM GENERATED SQL ---")
            print(clean_sql)
            print("LLM Data:")
            print(llm_df.to_string(index=False))
            print("=" * 60)

# Run inspection
debug_failed_cases(GOLDEN_BENCHMARK, llm, prompt_context)


❌ FAILED: [Q2_MEDIUM_SYLHET_TOP_ITEMS]
❓ Question: What are the top 3 items by total revenue in Sylhet?

--- GROUND TRUTH SQL ---
SELECT item_dim.item_name, SUM(fact_table.total_price) AS total_revenue
            FROM fact_table
            JOIN item_dim ON fact_table.item_key = item_dim.item_key
            JOIN store_dim ON fact_table.store_key = store_dim.store_key
            WHERE UPPER(store_dim.division) = 'SYLHET'
            GROUP BY item_dim.item_name
            ORDER BY total_revenue DESC
            LIMIT 3;
Ground Truth Data:
                          item_name  total_revenue
K Cups Daily Chef Columbian Supremo        69165.0
                      Red Bull 12oz        64130.0
                    Honey Packets          58455.0

--- LLM GENERATED SQL ---
SELECT item_dim.item_name, SUM(fact_table.total_price) AS total_revenue
FROM fact_table
JOIN item_dim ON fact_table.item_key = item_dim.item_key
JOIN store_dim ON fact_table.store_key = store_dim.store_key
WHERE UPPER(sto

### Testing across two different models and zero shot vs few shot prompts

In [31]:
# =============================================================================
# CHUNK 4D: FEW-SHOT EXEMPLAR PROMPT STRATEGY
# =============================================================================

FEW_SHOT_EXAMPLES = """
### EXEMPLAR EXAMPLES (Learn from these query patterns):

Example 1:
Question: "What is the total revenue for each product category?"
SQL:
SELECT item_dim.desc, SUM(fact_table.total_price) AS total_revenue
FROM fact_table
JOIN item_dim ON fact_table.item_key = item_dim.item_key
GROUP BY item_dim.desc
ORDER BY total_revenue DESC;

Example 2:
Question: "How many total transactions were made using cash?"
SQL:
SELECT COUNT(fact_table.payment_key) AS cash_transactions
FROM fact_table
JOIN trans_dim ON fact_table.payment_key = trans_dim.payment_key
WHERE UPPER(trans_dim.trans_type) = 'CASH';

Example 3:
Question: "Show top 5 stores by sale volume in 2017"
SQL:
SELECT store_dim.store_key, store_dim.division, COUNT(*) as total_sales
FROM fact_table
JOIN store_dim ON fact_table.store_key = store_dim.store_key
JOIN time_dim ON fact_table.time_key = time_dim.time_key
WHERE time_dim.year = 2017
GROUP BY store_dim.store_key, store_dim.division
ORDER BY total_sales DESC
LIMIT 5;
"""


def build_few_shot_context(base_context: str) -> str:
    """Inject exemplar question/SQL pairs into the base prompt context."""
    return base_context + "\n\n" + FEW_SHOT_EXAMPLES

In [35]:
# =============================================================================
# CHUNK 4E: AUTOMATED EXPERIMENT MATRIX BENCHMARK
# =============================================================================

import time
from langchain_groq import ChatGroq


def run_experiment_matrix(
    models: list,
    prompt_strategies: dict,
    test_cases: list,
    base_context: str
) -> pd.DataFrame:
    """
    Run an automated matrix experiment testing combinations of:
      - Models (e.g., openai/gpt-oss-120b vs openai/gpt-oss-20b)
      - Prompt Strategies (Zero-Shot vs Few-Shot)
    
    Measures: Execution Rate, Matching Accuracy, and Latency.
    """
    experiment_results = []
    
    for model_name in models:
        for strategy_name, context_text in prompt_strategies.items():
            print(f"\n🧪 Running Matrix Config: Model={model_name} | Strategy={strategy_name}")
            print("-" * 70)
            
            # Initialize Model
            exp_llm = ChatGroq(model_name=model_name, temperature=0.0)
            
            start_time = time.time()
            exec_passes = 0
            match_passes = 0
            total_tests = len(test_cases)
            
            for case in test_cases:
                q = case["question"]
                truth_sql = case["ground_truth_sql"]
                
                # Handle Guardrail
                if truth_sql == "BLOCKED":
                    res = ask_database(q, exp_llm, context_text)
                    if res.get("status") == "SECURITY_ERROR" or not res.get("success"):
                        exec_passes += 1
                        match_passes += 1
                    continue
                
                # Execute
                try:
                    truth_df = execute_sql(truth_sql)
                    res = ask_database(q, exp_llm, context_text)
                    
                    if res.get("success"):
                        exec_passes += 1
                        if compare_dataframes(res.get("results"), truth_df):
                            match_passes += 1
                except Exception:
                    pass
            
            elapsed_time = time.time() - start_time
            avg_latency = elapsed_time / total_tests
            
            experiment_results.append({
                "Model": model_name,
                "Prompt Strategy": strategy_name,
                "Execution Rate (%)": round((exec_passes / total_tests) * 100, 1),
                "Accuracy (%)": round((match_passes / total_tests) * 100, 1),
                "Total Time (s)": round(elapsed_time, 2),
                "Avg Latency/Q (s)": round(avg_latency, 2)
            })
            
    matrix_df = pd.DataFrame(experiment_results)
    
    print("\n" + "=" * 80)
    print("🏆 EXPERIMENTAL MATRIX BENCHMARK RESULTS")
    print("=" * 80)
    print(matrix_df.to_string(index=False))
    print("=" * 80 + "\n")
    
    return matrix_df


# --- Run Matrix Benchmark ---
prompt_strategies = {
    "Zero-Shot": prompt_context,
    "Few-Shot": build_few_shot_context(prompt_context)
}

models_to_test = [
    "openai/gpt-oss-20b",       # Lightweight & Fast
    "openai/gpt-oss-120b"   # High Intelligence
]

benchmark_matrix = run_experiment_matrix(
    models=models_to_test,
    prompt_strategies=prompt_strategies,
    test_cases=GOLDEN_BENCHMARK,
    base_context=prompt_context
)


🧪 Running Matrix Config: Model=openai/gpt-oss-20b | Strategy=Zero-Shot
----------------------------------------------------------------------

❓ Question: What is the total sales revenue across all stores?
--------------------------------------------------
📝 SQL Query:
SELECT SUM(total_price) AS total_sales_revenue
FROM fact_table

💡 Answer:
The total sales revenue across all stores is about $105.4 million.

❓ Question: What are the top 3 items by total revenue in Sylhet?
--------------------------------------------------
📝 SQL Query:
SELECT item_dim.item_name, SUM(fact_table.total_price) AS total_revenue
FROM fact_table
JOIN item_dim ON fact_table.item_key = item_dim.item_key
JOIN store_dim ON fact_table.store_key = store_dim.store_key
WHERE UPPER(store_dim.division) = 'SYLHET'
GROUP BY item_dim.item_name
ORDER BY total_revenue DESC
LIMIT 3

💡 Answer:
In Sylhet, the three best‑selling items by revenue are:

1. **K Cups Daily Chef Columbian Supremo** – $69,165  
2. **Red Bull 12 oz** 

### Testing 20 queries

In [36]:
# =============================================================================
# 20 SIMPLE-TO-MEDIUM GOLDEN QUERIES
# =============================================================================

GOLDEN_20 = [
    # --- Easy (1 table) ---
    {"id": "E1", "q": "How many total sales records are there?",
     "sql": "SELECT COUNT(*) AS total_rows FROM fact_table;"},
    {"id": "E2", "q": "What is the total revenue from all sales?",
     "sql": "SELECT SUM(total_price) AS total_revenue FROM fact_table;"},
    {"id": "E3", "q": "What is the average unit price in the fact table?",
     "sql": "SELECT AVG(unit_price) AS avg_unit_price FROM fact_table;"},
    {"id": "E4", "q": "How many unique items are sold?",
     "sql": "SELECT COUNT(DISTINCT item_key) AS unique_items FROM fact_table;"},
    {"id": "E5", "q": "What is the maximum total_price of a single sale?",
     "sql": "SELECT MAX(total_price) AS max_sale FROM fact_table;"},

    # --- Easy-Medium (1 join) ---
    {"id": "EM1", "q": "How many transactions were paid in cash?",
     "sql": "SELECT COUNT(*) AS cash_count FROM fact_table JOIN trans_dim ON fact_table.payment_key = trans_dim.payment_key WHERE UPPER(trans_dim.trans_type) = 'CASH';"},
    {"id": "EM2", "q": "How many stores are in the Dhaka division?",
     "sql": "SELECT COUNT(*) AS store_count FROM store_dim WHERE UPPER(division) = 'DHAKA';"},
    {"id": "EM3", "q": "List all unique payment types.",
     "sql": "SELECT DISTINCT trans_type FROM trans_dim;"},
    {"id": "EM4", "q": "How many customers are in the customer table?",
     "sql": "SELECT COUNT(*) AS customer_count FROM customer_dim;"},
    {"id": "EM5", "q": "What are the unique divisions in store_dim?",
     "sql": "SELECT DISTINCT division FROM store_dim;"},

    # --- Medium (2 joins / aggregation) ---
    {"id": "M1", "q": "What is the total revenue in Sylhet?",
     "sql": "SELECT SUM(fact_table.total_price) AS total_revenue FROM fact_table JOIN store_dim ON fact_table.store_key = store_dim.store_key WHERE UPPER(store_dim.division) = 'SYLHET';"},
    {"id": "M2", "q": "What is the total revenue in Dhaka?",
     "sql": "SELECT SUM(fact_table.total_price) AS total_revenue FROM fact_table JOIN store_dim ON fact_table.store_key = store_dim.store_key WHERE UPPER(store_dim.division) = 'DHAKA';"},
    {"id": "M3", "q": "Top 3 items by revenue overall.",
     "sql": "SELECT item_dim.item_name, SUM(fact_table.total_price) AS total_revenue FROM fact_table JOIN item_dim ON fact_table.item_key = item_dim.item_key GROUP BY item_dim.item_name ORDER BY total_revenue DESC LIMIT 3;"},
    {"id": "M4", "q": "Total quantity sold for each division.",
     "sql": "SELECT store_dim.division, SUM(fact_table.quantity) AS total_qty FROM fact_table JOIN store_dim ON fact_table.store_key = store_dim.store_key GROUP BY store_dim.division ORDER BY total_qty DESC;"},
    {"id": "M5", "q": "Total revenue by payment type.",
     "sql": "SELECT trans_dim.trans_type, SUM(fact_table.total_price) AS total_revenue FROM fact_table JOIN trans_dim ON fact_table.payment_key = trans_dim.payment_key GROUP BY trans_dim.trans_type ORDER BY total_revenue DESC;"},
    {"id": "M6", "q": "How many sales happened in 2017?",
     "sql": "SELECT COUNT(*) AS sales_2017 FROM fact_table JOIN time_dim ON fact_table.time_key = time_dim.time_key WHERE time_dim.year = 2017;"},
    {"id": "M7", "q": "Total revenue in 2017.",
     "sql": "SELECT SUM(fact_table.total_price) AS revenue_2017 FROM fact_table JOIN time_dim ON fact_table.time_key = time_dim.time_key WHERE time_dim.year = 2017;"},
    {"id": "M8", "q": "Top 3 divisions by total revenue.",
     "sql": "SELECT store_dim.division, SUM(fact_table.total_price) AS total_revenue FROM fact_table JOIN store_dim ON fact_table.store_key = store_dim.store_key GROUP BY store_dim.division ORDER BY total_revenue DESC LIMIT 3;"},
    {"id": "M9", "q": "Average quantity sold per transaction in Khulna.",
     "sql": "SELECT AVG(fact_table.quantity) AS avg_qty FROM fact_table JOIN store_dim ON fact_table.store_key = store_dim.store_key WHERE UPPER(store_dim.division) = 'KHULNA';"},
    {"id": "M10", "q": "How many unique items were sold in Sylhet?",
     "sql": "SELECT COUNT(DISTINCT fact_table.item_key) AS unique_items FROM fact_table JOIN store_dim ON fact_table.store_key = store_dim.store_key WHERE UPPER(store_dim.division) = 'SYLHET';"},
]

print(f"✅ Loaded {len(GOLDEN_20)} golden queries.")

✅ Loaded 20 golden queries.


In [ ]:
# =============================================================================
# FAST EVALUATOR — SQL only, no answer synthesis
# =============================================================================
import time
import numpy as np
import pandas as pd
from langchain_groq import ChatGroq


def fast_compare(df_a: pd.DataFrame, df_b: pd.DataFrame) -> bool:
    """Robust, quiet dataframe compare (Pandas 4.0 ready)."""
    if df_a is None or df_b is None:
        return False
    if df_a.empty and df_b.empty:
        return True
    if df_a.shape != df_b.shape:
        return False
    try:
        a, b = df_a.copy(), df_b.copy()
        str_dtypes = ["object", "str"]

        for d in (a, b):
            # Include both 'object' and 'str' to silence Pandas warnings
            for c in d.select_dtypes(include=str_dtypes).columns:
                d[c] = d[c].astype(str).str.strip().str.upper()
            for c in d.select_dtypes(include="number").columns:
                d[c] = d[c].round(2)

        a = a.sort_values(by=list(a.columns)).reset_index(drop=True)
        b = b.sort_values(by=list(b.columns)).reset_index(drop=True)

        num_ok = True
        if len(a.select_dtypes(include="number").columns):
            num_ok = np.allclose(
                a.select_dtypes(include="number").values,
                b.select_dtypes(include="number").values,
                rtol=1e-2, equal_nan=True
            )

        obj_ok = True
        if len(a.select_dtypes(include=str_dtypes).columns):
            obj_ok = np.array_equal(
                a.select_dtypes(include=str_dtypes).values,
                b.select_dtypes(include=str_dtypes).values
            )

        return num_ok and obj_ok
    except Exception:
        return False

def eval_one(question: str, truth_sql: str, llm, context: str) -> dict:
    """Generate → sanitize → execute → compare. No synthesis."""
    t0 = time.time()
    try:
        raw = generate_sql(question, llm, context)
        sql = sanitize_and_validate_sql(raw)
        pred_df = execute_sql(sql)
        truth_df = execute_sql(truth_sql)
        match = fast_compare(pred_df, truth_df)
        return {
            "executed": True,
            "match": match,
            "latency": round(time.time() - t0, 2),
            "gen_sql": sql,
            "error": None,
        }
    except Exception as e:
        return {
            "executed": False,
            "match": False,
            "latency": round(time.time() - t0, 2),
            "gen_sql": None,
            "error": str(e)[:120],
        }

In [38]:
# =============================================================================
# 2×2 EXPERIMENT ON 20 QUERIES
# =============================================================================

FEW_SHOT_EXAMPLES = """
### EXAMPLES:
Q: How many transactions were paid in cash?
SQL: SELECT COUNT(*) AS cash_count FROM fact_table JOIN trans_dim ON fact_table.payment_key = trans_dim.payment_key WHERE UPPER(trans_dim.trans_type) = 'CASH';

Q: What is the total revenue in Sylhet?
SQL: SELECT SUM(fact_table.total_price) AS total_revenue FROM fact_table JOIN store_dim ON fact_table.store_key = store_dim.store_key WHERE UPPER(store_dim.division) = 'SYLHET';

Q: Top 3 items by revenue.
SQL: SELECT item_dim.item_name, SUM(fact_table.total_price) AS total_revenue FROM fact_table JOIN item_dim ON fact_table.item_key = item_dim.item_key GROUP BY item_dim.item_name ORDER BY total_revenue DESC LIMIT 3;
"""

def run_20x4(base_context: str, queries: list = GOLDEN_20) -> pd.DataFrame:
    """
    4 configs:
      1. openai/gpt-oss-20b  + Zero-Shot
      2. openai/gpt-oss-20b  + Few-Shot
      3. openai/gpt-oss-120b + Zero-Shot
      4. openai/gpt-oss-120b+ Few-Shot
    """
    configs = [
        ("openai/gpt-oss-20b",    "Zero-Shot", base_context),
        ("openai/gpt-oss-20b",    "Few-Shot",  base_context + "\n" + FEW_SHOT_EXAMPLES),
        ("openai/gpt-oss-120b", "Zero-Shot", base_context),
        ("openai/gpt-oss-120b", "Few-Shot",  base_context + "\n" + FEW_SHOT_EXAMPLES),
    ]

    rows = []
    for model_name, strategy, ctx in configs:
        print(f"\n🧪 {model_name} | {strategy}")
        llm = ChatGroq(model_name=model_name, temperature=0.0, max_tokens=512)

        exec_ok = match_ok = 0
        latencies = []

        for i, case in enumerate(queries, 1):
            r = eval_one(case["q"], case["sql"], llm, ctx)
            exec_ok += int(r["executed"])
            match_ok += int(r["match"])
            latencies.append(r["latency"])
            status = "✅" if r["match"] else ("⚠️" if r["executed"] else "❌")
            print(f"  [{i:02d}/20] {status} {case['id']} ({r['latency']}s)")

        n = len(queries)
        rows.append({
            "Model": model_name.replace("llama-", ""),
            "Strategy": strategy,
            "Exec %": round(100 * exec_ok / n, 1),
            "Accuracy %": round(100 * match_ok / n, 1),
            "Avg Latency (s)": round(sum(latencies) / n, 2),
        })

    result = pd.DataFrame(rows)
    print("\n" + "=" * 70)
    print("🏆 FINAL 20-QUERY BENCHMARK")
    print("=" * 70)
    print(result.to_string(index=False))
    print("=" * 70)
    return result


# --- Run it ---
benchmark_df = run_20x4(prompt_context)


🧪 openai/gpt-oss-20b | Zero-Shot
  [01/20] ✅ E1 (0.86s)
  [02/20] ✅ E2 (0.86s)
  [03/20] ✅ E3 (0.82s)
  [04/20] ✅ E4 (1.84s)
  [05/20] ✅ E5 (0.89s)
  [06/20] ✅ EM1 (2.73s)
  [07/20] ✅ EM2 (7.46s)


C:\Users\harry\AppData\Local\Temp\ipykernel_33128\2508438185.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in d.select_dtypes(include="object").columns:
C:\Users\harry\AppData\Local\Temp\ipykernel_33128\2508438185.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/m

  [08/20] ✅ EM3 (0.98s)
  [09/20] ✅ EM4 (0.55s)


C:\Users\harry\AppData\Local\Temp\ipykernel_33128\2508438185.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in d.select_dtypes(include="object").columns:
C:\Users\harry\AppData\Local\Temp\ipykernel_33128\2508438185.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/m

  [10/20] ✅ EM5 (1.33s)
  [11/20] ✅ M1 (22.21s)
  [12/20] ⚠️ M2 (24.54s)


...

  [14/20] ✅ M4 (2.5s)


C:\Users\harry\AppData\Local\Temp\ipykernel_33128\2508438185.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in d.select_dtypes(include="object").columns:
C:\Users\harry\AppData\Local\Temp\ipykernel_33128\2508438185.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/m

  [15/20] ✅ M5 (2.08s)
  [16/20] ✅ M6 (18.35s)
  [17/20] ✅ M7 (21.61s)


C:\Users\harry\AppData\Local\Temp\ipykernel_33128\2508438185.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in d.select_dtypes(include="object").columns:
C:\Users\harry\AppData\Local\Temp\ipykernel_33128\2508438185.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/m

  [18/20] ✅ M8 (3.5s)
  [19/20] ⚠️ M9 (22.07s)
  [20/20] ✅ M10 (21.56s)

🏆 FINAL 20-QUERY BENCHMARK
              Model  Strategy  Exec %  Accuracy %  Avg Latency (s)
 openai/gpt-oss-20b Zero-Shot   100.0        90.0             8.16
 openai/gpt-oss-20b  Few-Shot   100.0        95.0             7.95
openai/gpt-oss-120b Zero-Shot   100.0        90.0             7.86
openai/gpt-oss-120b  Few-Shot   100.0        95.0             8.10


In [41]:
# =============================================================================
# INSPECT ZERO-SHOT FAILURES
# =============================================================================

def find_zero_shot_failures(queries: list, llm, context: str):
    """Prints which queries failed in Zero-Shot mode."""
    print("🔍 Inspecting Zero-Shot Failures...\n")
    failed_count = 0
    
    for case in queries:
        raw_sql = generate_sql(case["q"], llm, context)
        clean_sql = sanitize_and_validate_sql(raw_sql)
        
        pred_df = execute_sql(clean_sql)
        truth_df = execute_sql(case["sql"])
        
        if not fast_compare(pred_df, truth_df):
            failed_count += 1
            print(f"❌ Failure #{failed_count} | ID: {case['id']}")
            print(f"❓ Question: {case['q']}")
            print(f"🎯 Expected SQL: {case['sql']}")
            print(f"🤖 LLM SQL:     {clean_sql}\n" + "-"*60)

# Run on Zero-Shot context (without few-shot examples)
find_zero_shot_failures(GOLDEN_20, llm, prompt_context)

🔍 Inspecting Zero-Shot Failures...



C:\Users\harry\AppData\Local\Temp\ipykernel_33128\2508438185.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in d.select_dtypes(include="object").columns:
C:\Users\harry\AppData\Local\Temp\ipykernel_33128\2508438185.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/m

❌ Failure #1 | ID: M6
❓ Question: How many sales happened in 2017?
🎯 Expected SQL: SELECT COUNT(*) AS sales_2017 FROM fact_table JOIN time_dim ON fact_table.time_key = time_dim.time_key WHERE time_dim.year = 2017;
🤖 LLM SQL:     SELECT COUNT(DISTINCT fact_table.payment_key) AS sales_count
FROM fact_table
JOIN time_dim ON fact_table.time_key = time_dim.time_key
WHERE time_dim.year = 2017
------------------------------------------------------------


C:\Users\harry\AppData\Local\Temp\ipykernel_33128\2508438185.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in d.select_dtypes(include="object").columns:
C:\Users\harry\AppData\Local\Temp\ipykernel_33128\2508438185.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/m

❌ Failure #2 | ID: M9
❓ Question: Average quantity sold per transaction in Khulna.
🎯 Expected SQL: SELECT AVG(fact_table.quantity) AS avg_qty FROM fact_table JOIN store_dim ON fact_table.store_key = store_dim.store_key WHERE UPPER(store_dim.division) = 'KHULNA';
🤖 LLM SQL:     SELECT AVG(total_qty) AS avg_quantity_per_transaction
FROM (
    SELECT fact_table.payment_key,
           SUM(fact_table.quantity) AS total_qty
    FROM fact_table
    JOIN store_dim
      ON fact_table.store_key = store_dim.store_key
    WHERE UPPER(store_dim.division) = 'KHULNA'
    GROUP BY fact_table.payment_key
)
------------------------------------------------------------


In [39]:
# =============================================================================
# STEP A: FIND THE 1 FAILING QUERY
# =============================================================================

def find_failing_query(queries: list, llm, context: str):
    """Prints the exact query where the LLM's answer didn't match ground truth."""
    for case in queries:
        raw_sql = generate_sql(case["q"], llm, context)
        clean_sql = sanitize_and_validate_sql(raw_sql)
        
        pred_df = execute_sql(clean_sql)
        truth_df = execute_sql(case["sql"])
        
        if not fast_compare(pred_df, truth_df):
            print(f"❌ Failed Query ID: {case['id']}")
            print(f"❓ Question: {case['q']}")
            print(f"🎯 Ground Truth SQL:\n   {case['sql']}")
            print(f"🤖 LLM Generated SQL:\n   {clean_sql}\n")

# Run on the Few-Shot context
few_shot_ctx = prompt_context + "\n" + FEW_SHOT_EXAMPLES
find_failing_query(GOLDEN_20, llm, few_shot_ctx)

C:\Users\harry\AppData\Local\Temp\ipykernel_33128\2508438185.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in d.select_dtypes(include="object").columns:
C:\Users\harry\AppData\Local\Temp\ipykernel_33128\2508438185.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/m

❌ Failed Query ID: M9
❓ Question: Average quantity sold per transaction in Khulna.
🎯 Ground Truth SQL:
   SELECT AVG(fact_table.quantity) AS avg_qty FROM fact_table JOIN store_dim ON fact_table.store_key = store_dim.store_key WHERE UPPER(store_dim.division) = 'KHULNA';
🤖 LLM Generated SQL:
   SELECT AVG(total_qty) AS avg_qty_per_transaction
FROM (
  SELECT fact_table.payment_key, SUM(fact_table.quantity) AS total_qty
  FROM fact_table
  JOIN store_dim ON fact_table.store_key = store_dim.store_key
  WHERE UPPER(store_dim.division) = 'KHULNA'
  GROUP BY fact_table.payment_key
)



In [ ]:
    # =============================================================================
    # STEP B: TEMPERATURE SENSITIVITY TEST
    # =============================================================================

from langchain_groq import ChatGroq

def test_temperature_impact(queries: list, context: str):
    """Compare deterministic (temp=0.0) vs creative (temp=0.7) SQL generation."""
    
    temps = [0.0, 0.7]
    results = []
    
    for temp in temps:
        print(f"🌡️ Testing Temperature = {temp}...")
        test_llm = ChatGroq(model_name="openai/gpt-oss-120b", temperature=temp)
        
        correct = 0
        for case in queries:
            try:
                raw_sql = generate_sql(case["q"], test_llm, context)
                sql = sanitize_and_validate_sql(raw_sql)
                pred_df = execute_sql(sql)
                truth_df = execute_sql(case["sql"])
                if fast_compare(pred_df, truth_df):
                    correct += 1
            except Exception:
                pass  # Execution or security failure
                
        acc = (correct / len(queries)) * 100
        results.append({"Temperature": temp, "Accuracy (%)": acc})
        
    print("\n" + "="*40)
    print("🌡️ TEMPERATURE TEST RESULTS")
    print("="*40)
    print(pd.DataFrame(results).to_string(index=False))

# Run test
test_temperature_impact(GOLDEN_20, few_shot_ctx)

🌡️ Testing Temperature = 0.0...


C:\Users\harry\AppData\Local\Temp\ipykernel_33128\2508438185.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in d.select_dtypes(include="object").columns:
C:\Users\harry\AppData\Local\Temp\ipykernel_33128\2508438185.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/m

🌡️ Testing Temperature = 0.7...


C:\Users\harry\AppData\Local\Temp\ipykernel_33128\2508438185.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in d.select_dtypes(include="object").columns:
C:\Users\harry\AppData\Local\Temp\ipykernel_33128\2508438185.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/m


🌡️ TEMPERATURE TEST RESULTS
 Temperature  Accuracy (%)
         0.0          95.0
         0.7          90.0


In [42]:
# =============================================================================
# REAL-WORLD EVALUATION: ACCURACY BY DIFFICULTY & LATENCY
# =============================================================================

import time
import pandas as pd

# Categorize our 20 queries into Easy, Medium, Hard
DIFFICULTY_MAPPING = {
    "E1": "Easy", "E2": "Easy", "E3": "Easy", "E4": "Easy", "E5": "Easy",
    "EM1": "Medium", "EM2": "Medium", "EM3": "Easy", "EM4": "Easy", "EM5": "Easy",
    "M1": "Medium", "M2": "Medium", "M3": "Medium", "M4": "Medium", "M5": "Medium",
    "M6": "Medium", "M7": "Medium", "M8": "Medium", "M9": "Hard", "M10": "Medium"
}


def evaluate_by_difficulty(queries: list, llm, context: str) -> pd.DataFrame:
    """Evaluates pipeline performance grouped by query difficulty."""
    records = []
    
    for case in queries:
        qid = case["id"]
        difficulty = DIFFICULTY_MAPPING.get(qid, "Medium")
        
        t0 = time.time()
        try:
            raw_sql = generate_sql(case["q"], llm, context)
            clean_sql = sanitize_and_validate_sql(raw_sql)
            pred_df = execute_sql(clean_sql)
            truth_df = execute_sql(case["sql"])
            
            is_match = fast_compare(pred_df, truth_df)
            latency = time.time() - t0
        except Exception:
            is_match = False
            latency = time.time() - t0
            
        records.append({
            "ID": qid,
            "Difficulty": difficulty,
            "Match": is_match,
            "Latency": latency
        })
        
    df = pd.DataFrame(records)
    
    # Group by Difficulty
    summary = df.groupby("Difficulty").agg(
        Total_Queries=("Match", "count"),
        Accuracy_Pct=("Match", lambda x: round((x.sum() / x.count()) * 100, 1)),
        Avg_Latency_Sec=("Latency", lambda x: round(x.mean(), 2))
    ).reset_index()
    
    return summary

# --- Run on your current LLM ---
diff_report = evaluate_by_difficulty(GOLDEN_20, llm, prompt_context)

print("\n" + "="*60)
print("📊 ACCURACY STRATIFIED BY DIFFICULTY")
print("="*60)
print(diff_report.to_string(index=False))
print("="*60)

C:\Users\harry\AppData\Local\Temp\ipykernel_33128\2508438185.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  for c in d.select_dtypes(include="object").columns:
C:\Users\harry\AppData\Local\Temp\ipykernel_33128\2508438185.py:21: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/m


📊 ACCURACY STRATIFIED BY DIFFICULTY
Difficulty  Total_Queries  Accuracy_Pct  Avg_Latency_Sec
      Easy              8         100.0             1.42
      Hard              1           0.0            75.01
    Medium             11          90.9            30.05
